# Guide 5 — Cell descriptors

Cell descriptors go beyond datasheet-level specifications. They capture electrode composition, electrolyte formulation, separator properties, and cell construction — the detail needed for research-grade records and curated cell libraries.

**Estimated time:** 40 minutes
**Prerequisites:** [Guide 2 — Describing a cell](02-first-cell-type.ipynb)

Everything writes under `.battinfo/notebooks/guide-05`.

In [ ]:
import os, json, warnings
from pathlib import Path

_here = Path.cwd()
if (_here / "src" / "battinfo").exists():
    REPO = _here
elif (_here.parent.parent / "src" / "battinfo").exists():
    REPO = _here.parent.parent
    os.chdir(REPO)
else:
    REPO = _here

print(f"Repo root: {REPO}")

WORKSPACE = REPO / '.battinfo/notebooks/guide-05'
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORKSPACE)

## Step 1: Individual materials

In [ ]:
from battinfo.authoring import material, properties

# Positive electrode
lfp   = material("LFP",          mass_fraction={"value": 90,  "unit": "%"},
                  comment="LiFePO4, olivine structure, D50 ~ 3 µm (Aleees GEN2)")
pvdf  = material("PVDF",         mass_fraction={"value":  5,  "unit": "%"})
c65   = material("Carbon black", mass_fraction={"value":  5,  "unit": "%"},
                  comment="Imerys C65")

# Negative electrode
graphite = material("Graphite",  mass_fraction={"value": 95.5, "unit": "%"},
                     comment="Natural graphite, D50 ~ 20 µm")
cmc  = material("CMC", mass_fraction={"value": 1.5, "unit": "%"})
sbr  = material("SBR", mass_fraction={"value": 2.5, "unit": "%"})
c_ng = material("Carbon black", mass_fraction={"value": 0.5, "unit": "%"})

print("Materials defined.")

## Step 2: Bills of materials

In [ ]:
from battinfo.authoring import bom

positive_bom = bom(active_material=lfp,      binder=pvdf,       additive=c65)
negative_bom = bom(active_material=graphite, binder=[cmc, sbr], additive=c_ng)

print("Positive active:", positive_bom.active_material[0].name)
print("Negative binders:", [m.name for m in negative_bom.binder])

## Step 3: Electrodes

In [ ]:
from battinfo.authoring import electrode

positive_electrode = electrode(
    bom=positive_bom,
    loading={"value": 12.5, "unit": "mg/cm2"},
    calendered_density={"value": 2.4, "unit": "g/cm3"},
    current_collector="Aluminium foil",
    current_collector_thickness={"value": 16.0, "unit": "µm"},
    coating_comment="NMP-cast, roll-calendered to 85% theoretical density",
)

negative_electrode = electrode(
    bom=negative_bom,
    loading={"value": 7.8, "unit": "mg/cm2"},
    calendered_density={"value": 1.55, "unit": "g/cm3"},
    current_collector="Copper foil",
    current_collector_thickness={"value": 10.0, "unit": "µm"},
)

print("Positive loading:", positive_electrode.coating.property.get("loading"))
print("Negative density:", negative_electrode.coating.property.get("calendered_density"))

## Step 4: Electrolyte

In [ ]:
from battinfo.authoring import electrolyte_recipe, material as mat

lp30 = electrolyte_recipe(
    family="organic",
    salt="LiPF6",
    salt_concentration={"value": 1.0, "unit": "mol/L"},
    solvents=[
        mat("EC",  volume_fraction={"value": 50, "unit": "%"}),
        mat("DMC", volume_fraction={"value": 50, "unit": "%"}),
    ],
    additives=[
        mat("VC",  mass_fraction={"value": 2.0, "unit": "%"}, comment="SEI-forming additive"),
        mat("FEC", mass_fraction={"value": 1.0, "unit": "%"}, comment="Fluoroethylene carbonate"),
    ],
    comment="LP30 + 2% VC + 1% FEC",
)

print("Salt:", lp30.salt.name, "at", lp30.salt.property.get("concentration"))
print("Solvents:", [s.name for s in lp30.solvent_mixture.component])
print("Additives:", [a.name for a in lp30.additive])

## Step 5: Separator

In [ ]:
from battinfo.authoring import separator_spec

sep = separator_spec(
    material="Polypropylene",
    thickness={"value": 25.0, "unit": "µm"},
    properties=properties(porosity={"value": 41, "unit": "%"}),
    comment="Celgard 2500, single-layer PP",
)

print("Material:", sep.material)
print("Thickness:", sep.property.get("thickness"))
print("Porosity:", sep.property.get("porosity"))

## Step 6: Construction

In [ ]:
from battinfo.authoring import construction

build = construction(
    assembly_type="wound",
    layering="jelly-roll",
    layer_count=1,
    comment="Single jelly-roll, wound on 4 mm mandrel",
)

print("Assembly:", build.assembly_type, "/", build.layering)

## Step 7: Provenance

In [ ]:
from battinfo.authoring import source

prov = source(
    type="datasheet",
    name="ANR26650M1-B Product Datasheet Rev B",
    url="https://example.com/a123-anr26650m1-b.pdf",
    retrieved_at=1746230400,
    curated_by="Jane Smith",
)

print("Source type:", prov.type)
print("Curated by:", prov.curated_by)

## Step 8: Cell description

In [ ]:
from battinfo.authoring import cell_description

# Use the known IRI for the A123 ANR26650M1-B example cell spec.
# For new records: publish a CellSpecification first to get the IRI, then use it here.
A123_IRI = "https://w3id.org/battinfo/spec/7d9k-2m4p-8t3x-6nq5"

spec = cell_description(
    id=A123_IRI,
    manufacturer="A123 Systems",
    model="ANR26650M1-B",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="LFP",
    negative_electrode_basis="graphite",
    size_code="R26650",
    positive_electrode=positive_electrode,
    negative_electrode=negative_electrode,
    electrolyte=lp30,
    separator=sep,
    construction=build,
    source=prov,
    comment=[
        "Descriptor curated from manufacturer datasheet and published literature.",
        "Electrode loadings estimated from capacity balance and assumed density.",
    ],
)

print("Cell ID:", spec.id)
print("Model:", spec.model)

## Step 9: Serialise the descriptor

In [ ]:
# Write the descriptor to disk for inspection and downstream use.
desc_dir = WORKSPACE / "descriptors"
desc_dir.mkdir(parents=True, exist_ok=True)
desc_path = desc_dir / "a123-anr26650m1-b.descriptor.json"
spec_dict = {"schema_version": "1.0.0", "specification": spec.to_json()}
desc_path.write_text(json.dumps(spec_dict, indent=2, ensure_ascii=False), encoding="utf-8")
print("Written:", desc_path.relative_to(WORKSPACE))
print("Top-level keys:", list(spec_dict.keys())[:8])

## Step 10: Map to domain-battery JSON-LD

In [ ]:
from battinfo.transform.json_to_jsonld import to_jsonld

descriptor_doc = json.loads(desc_path.read_text(encoding="utf-8"))
desc_jsonld = to_jsonld(descriptor_doc, target="domain-battery")

battery = desc_jsonld["@graph"][0]
print("@type:", battery["@type"])
print()
pos = battery.get("hasPositiveElectrode", {})
print("Positive electrode @type:", pos.get("@type"))
neg = battery.get("hasNegativeElectrode", {})
print("Negative electrode @type:", neg.get("@type"))

In [ ]:
# Export the domain-battery JSON-LD to disk
desc_jsonld_path = WORKSPACE / "descriptors" / "a123-anr26650m1-b.domain-battery.jsonld"
desc_jsonld_path.write_text(
    json.dumps(desc_jsonld, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Domain-battery JSON-LD exported to: {desc_jsonld_path.relative_to(WORKSPACE)}")
print()
print(json.dumps(desc_jsonld, indent=2))

## Step 11: Validate the descriptor JSON-LD

In [ ]:
from battinfo.validate.jsonld import validate_jsonld_report

# Validate the domain-battery JSON-LD produced from the descriptor
report = validate_jsonld_report(desc_jsonld)
print("ok:", report.ok, "| errors:", len(report.errors), "| warnings:", len(report.warnings))
for issue in report.issues:
    print(f"  [{issue.severity}] {issue.code}: {issue.message}")

## Step 12: Publish

In [ ]:
from battinfo import CellSpecification, publish
from battinfo.api import publish_record

ct = CellSpecification(
    manufacturer=spec.manufacturer, model=spec.model,
    format=spec.format, chemistry=spec.chemistry,
    positive_electrode_basis=spec.positive_electrode_basis,
    negative_electrode_basis=spec.negative_electrode_basis,
)

pub = publish(ct, destination="local", root=WORKSPACE / "published")
print("IRI:", pub.canonical_iri)

In [ ]:
output = publish_record(
    pub.debug_paths["canonical_record_path"],
    target_root=WORKSPACE / "resolver",
)
jsonld_out = json.loads(
    Path(output["output_dir"], "index.jsonld").read_text(encoding="utf-8")
)

print("@type:", jsonld_out["@type"])
print()
print("Properties:")
for prop in jsonld_out.get("hasProperty", []):
    t = prop["@type"][0] if isinstance(prop["@type"], list) else prop["@type"]
    v = prop.get("hasNumericalPart", {}).get("hasNumericalValue", "—")
    print(f"  {t}: {v}")

## Next

Return to **[Guide 3 — Linked records](03-linked-records.ipynb)** to connect this descriptor cell spec to instances, tests, and datasets.